# Prediccion de Riesgo Academico en Matematica
## Evaluacion Muestral 2022 · MINEDU · Lima Metropolitana

**Proyecto P20261012** · UPC · Ingenieria de Sistemas de Informacion

- Dataset: EM 2022, 2.° Secundaria, Lima Metro, gestion privada
- **3 629 estudiantes · 84 colegios · 30 distritos**

---

### Objetivo
Entrenar un modelo que identifique estudiantes en riesgo de bajo rendimiento en Matematica
(`grupo_M in {Previo al inicio, En inicio}`) usando el puntaje de Lectura, Ciencias y
variables de contexto — sin usar el puntaje de Matematica como feature.

### Estructura del notebook
1. Configuracion e instalacion de dependencias
2. Carga del dataset
3. Analisis exploratorio (EDA)
4. Definicion de features y target
5. Division train/test sin leakage por colegio (GroupShuffleSplit)
6. Feature engineering — agregados por IE calculados solo en train
7. Comparativa de 4 modelos con GroupKFold(5)
8. Calibracion isotonica del modelo ganador
9. Analisis SHAP — importancia global y explicacion individual
10. Metricas finales en test (17 IEs nunca vistas)
11. Guardar artefactos .pkl

## 1. Configuracion

In [ ]:
# Instalar dependencias no incluidas en Colab por defecto
!pip install -q lightgbm xgboost shap
print('Dependencias listas.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve,
    precision_score, recall_score
)
import xgboost as xgb
import lightgbm as lgb

# Estilo global
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titleweight': 'semibold',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})
PALETA = {
    'azul': '#1f4e79', 'rojo': '#a63a3a', 'verde': '#3b6d4a',
    'naranja': '#d4843a', 'gris': '#6b6b6b', 'morado': '#6b4c9a'
}

SEED = 42
print('Librerias importadas correctamente.')

## 2. Carga del dataset

Archivo: `em_2022_lima_privado.csv` — dataset procesado a partir de los microdatos publicos
de la Evaluacion Muestral 2022 (UMC-MINEDU). Filtro aplicado: Lima Metropolitana + gestion No estatal.

**Como cargar en Colab:**
- **Opcion A (recomendada):** Ejecutar la celda de abajo y subir el CSV cuando se solicite.
- **Opcion B:** Montar Google Drive si el archivo esta guardado ahi (descomentar bloque Drive).

In [ ]:
CSV_NAME = 'em_2022_lima_privado.csv'

# --- Opcion A: Subida directa ---
if not os.path.exists(CSV_NAME):
    from google.colab import files
    print(f'Selecciona el archivo {CSV_NAME}:')
    uploaded = files.upload()
    for fname in uploaded:
        if fname != CSV_NAME:
            os.rename(fname, CSV_NAME)

# --- Opcion B: Desde Google Drive (descomentar si prefieres) ---
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# RUTA_DRIVE = '/content/drive/MyDrive/TesisDG-ML/em_2022_lima_privado.csv'
# shutil.copy(RUTA_DRIVE, CSV_NAME)

df = pd.read_csv(CSV_NAME)

# Asegurar tipos numericos (columnas pueden venir como object desde Excel original)
for col in ['M500_L', 'M500_CN', 'M500_M', 'ise']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=['M500_L', 'M500_CN', 'ise']).reset_index(drop=True)

print(f'Dataset cargado: {df.shape[0]:,} estudiantes | {df.shape[1]} columnas')
print(f'Colegios (ID_IE): {df["ID_IE"].nunique()} | Distritos: {df["Distrito"].nunique()}')
df.head()

## 3. Analisis exploratorio (EDA)

Variables disponibles:

| Variable | Tipo | Descripcion |
|---|---|---|
| `ID_IE` | ID | Identificador del colegio (usado para validacion sin leakage) |
| `sexo` | Categorica | Hombre / Mujer |
| `ise` | Numerica | Indice socioeconomico individual (aprox. -2 a +3) |
| `Distrito` | Categorica (30) | Distrito del colegio |
| `M500_L` | Numerica | Puntaje de Lectura (escala 500) |
| `M500_CN` | Numerica | Puntaje de Ciencia y Tecnologia (escala 500) |
| `M500_M` | Numerica | Puntaje de Matematica — **no se usa como feature** |
| `grupo_M` | Categorica | Nivel de logro en Matematica (4 niveles UMC) |
| `riesgo_matematica` | Binaria | **Target** — 1 si grupo_M in {Previo al inicio, En inicio} |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('EDA — EM 2022 Lima Metro Privado', fontsize=14, fontweight='bold', y=1.01)

# 1. Balance del target
tc = df['riesgo_matematica'].value_counts()
axes[0, 0].bar(['Sin riesgo (0)', 'En riesgo (1)'], tc.values,
               color=[PALETA['verde'], PALETA['rojo']], edgecolor='white', width=0.5)
for i, v in enumerate(tc.values):
    axes[0, 0].text(i, v + 15, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
axes[0, 0].set_title('Balance del target')
axes[0, 0].set_ylabel('Estudiantes')

# 2. ISE por nivel de riesgo
axes[0, 1].boxplot(
    [df[df.riesgo_matematica == 0]['ise'], df[df.riesgo_matematica == 1]['ise']],
    labels=['Sin riesgo', 'En riesgo'],
    patch_artist=True,
    boxprops=dict(facecolor='white', color=PALETA['azul']),
    medianprops=dict(color=PALETA['rojo'], linewidth=2.5),
    whiskerprops=dict(color=PALETA['gris']),
    capprops=dict(color=PALETA['gris']),
)
axes[0, 1].set_title('ISE por nivel de riesgo')
axes[0, 1].set_ylabel('ISE')

# 3. Distribucion M500_L
for nivel, color in [(0, PALETA['verde']), (1, PALETA['rojo'])]:
    axes[0, 2].hist(df[df.riesgo_matematica == nivel]['M500_L'],
                    bins=40, alpha=0.65, color=color,
                    label='Sin riesgo' if nivel == 0 else 'En riesgo')
axes[0, 2].set_title('Distribucion M500_L (Lectura)')
axes[0, 2].set_xlabel('Puntaje')
axes[0, 2].legend(frameon=False)

# 4. Distribucion M500_CN
for nivel, color in [(0, PALETA['verde']), (1, PALETA['rojo'])]:
    axes[1, 0].hist(df[df.riesgo_matematica == nivel]['M500_CN'],
                    bins=40, alpha=0.65, color=color,
                    label='Sin riesgo' if nivel == 0 else 'En riesgo')
axes[1, 0].set_title('Distribucion M500_CN (Ciencias)')
axes[1, 0].set_xlabel('Puntaje')
axes[1, 0].legend(frameon=False)

# 5. Tasa de riesgo por sexo
riesgo_sexo = df.groupby('sexo')['riesgo_matematica'].mean() * 100
bars = axes[1, 1].bar(riesgo_sexo.index, riesgo_sexo.values,
                      color=[PALETA['morado'], PALETA['naranja']], edgecolor='white', width=0.45)
for bar, val in zip(bars, riesgo_sexo.values):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, val + 0.5,
                    f'{val:.1f}%', ha='center', fontsize=11)
axes[1, 1].set_title('Tasa de riesgo por sexo')
axes[1, 1].set_ylabel('% en riesgo')
axes[1, 1].set_ylim(0, riesgo_sexo.max() * 1.25)

# 6. Top 10 distritos por tasa de riesgo
tasa_dist = (df.groupby('Distrito')['riesgo_matematica'].mean()
               .sort_values(ascending=True).tail(10) * 100)
tasa_dist.plot.barh(ax=axes[1, 2], color=PALETA['rojo'], edgecolor='white')
axes[1, 2].set_title('Top 10 distritos — mayor tasa de riesgo')
axes[1, 2].set_xlabel('% en riesgo')
for i, v in enumerate(tasa_dist.values):
    axes[1, 2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Tasa de riesgo global: {df["riesgo_matematica"].mean()*100:.1f}%')
print(f'Distribucion target: {dict(df["riesgo_matematica"].value_counts().sort_index())}')

In [ ]:
# Correlaciones entre features numericas y el target
num_cols = ['ise', 'M500_L', 'M500_CN', 'riesgo_matematica']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1,
            center=0, ax=ax, square=True, cbar_kws={'shrink': 0.75},
            xticklabels=num_cols, yticklabels=num_cols)
ax.set_title('Correlaciones — features numericas vs target')
plt.tight_layout()
plt.show()

## 4. Definicion de features y target

**Features del modelo (9 en total):**

| Grupo | Features | Justificacion |
|---|---|---|
| Categoricas | `sexo`, `Distrito` | Contexto demografico y geografico |
| Numericas individuales | `ise`, `M500_L`, `M500_CN` | ISE y puntajes correlacionados con matematica |
| Agregados por IE (calculados solo en train) | `M500_L_iemean`, `M500_CN_iemean`, `ise_iemean`, `tamanio_ie` | Efecto del colegio sobre el rendimiento |

**Por que NO se usa M500_M:** el puntaje de matematica es el target proxy. Incluirlo como feature
causaria leakage trivial (el modelo aprenderia a predecir el puntaje usando el puntaje mismo).

In [ ]:
FEATURES_CAT = ['sexo', 'Distrito']
FEATURES_NUM = ['ise', 'M500_L', 'M500_CN']
FEATURES_IE  = ['M500_L_iemean', 'M500_CN_iemean', 'ise_iemean', 'tamanio_ie']
ALL_FEATURES = FEATURES_CAT + FEATURES_NUM + FEATURES_IE
TARGET = 'riesgo_matematica'

print(f'Total features: {len(ALL_FEATURES)}')
print(f'  Categoricas:  {FEATURES_CAT}')
print(f'  Numericas:    {FEATURES_NUM}')
print(f'  Agregados IE: {FEATURES_IE}')

## 5. Division train / test sin leakage por colegio

**El problema del leakage por colegio:**  
Si estudiantes del mismo colegio caen en train y test, el modelo aprende el "efecto colegio"
y aparenta ser mas preciso de lo que realmente es al enfrentarse a un colegio nuevo.

**Solucion — GroupShuffleSplit:**  
Agrupa por `ID_IE`. Todos los alumnos de un colegio van al mismo set.  
El 20% de los **colegios** va al test — no el 20% de los alumnos.

```
ID_IE 001  → train  (todos sus alumnos)
ID_IE 002  → train
...
ID_IE 067  → train
ID_IE 068  → test   (nunca visto durante entrenamiento)
...
ID_IE 084  → test
```

In [ ]:
X_base = df[FEATURES_CAT + FEATURES_NUM].copy()
y = df[TARGET].copy()
groups = df['ID_IE'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X_base, y, groups))

train_ies = set(groups[train_idx])
test_ies  = set(groups[test_idx])

# Verificacion: cero solapamiento
assert len(train_ies & test_ies) == 0, 'LEAKAGE detectado!'

print(f'Train: {len(train_idx):,} estudiantes — {len(train_ies)} colegios')
print(f'Test:  {len(test_idx):,} estudiantes  — {len(test_ies)} colegios')
print(f'Solapamiento de IEs entre train y test: {len(train_ies & test_ies)} (cero = sin leakage)')
print(f'Tasa de riesgo en train: {y.iloc[train_idx].mean()*100:.1f}%')
print(f'Tasa de riesgo en test:  {y.iloc[test_idx].mean()*100:.1f}%')

## 6. Feature engineering — Agregados por IE

Se calculan las 4 variables a nivel de colegio usando **solo el conjunto de train**.  
Para los colegios del test que no aparecen en train, se usa la media global del train como fallback.

Este paso es critico: si se calcularan sobre todo el dataset, se estaria filtrando informacion
del test hacia el train (data leakage temporal).

In [ ]:
train_df = df.iloc[train_idx][['ID_IE', 'M500_L', 'M500_CN', 'ise']].copy()

ie_stats = train_df.groupby('ID_IE').agg(
    M500_L_iemean  = ('M500_L', 'mean'),
    M500_CN_iemean = ('M500_CN', 'mean'),
    ise_iemean     = ('ise', 'mean'),
    tamanio_ie     = ('M500_L', 'count'),
).reset_index()

global_means = {
    col: ie_stats[col].mean()
    for col in ['M500_L_iemean', 'M500_CN_iemean', 'ise_iemean', 'tamanio_ie']
}

def agregar_ie_stats(base_df, idx_array):
    out = base_df.copy()
    out['ID_IE'] = df['ID_IE'].iloc[idx_array].values
    out = out.merge(ie_stats, on='ID_IE', how='left').drop(columns='ID_IE')
    for col, val in global_means.items():
        out[col] = out[col].fillna(val)
    return out[ALL_FEATURES]

X_train_base = df.iloc[train_idx][FEATURES_CAT + FEATURES_NUM].copy()
X_test_base  = df.iloc[test_idx][FEATURES_CAT + FEATURES_NUM].copy()

X_train = agregar_ie_stats(X_train_base, train_idx)
X_test  = agregar_ie_stats(X_test_base, test_idx)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test  = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups[train_idx]

print(f'Shape X_train: {X_train.shape}')
print(f'Shape X_test:  {X_test.shape}')
print(f'Nulos en train: {X_train.isnull().sum().sum()} | Nulos en test: {X_test.isnull().sum().sum()}')
X_train.head()

## 7. Comparativa de modelos — GroupKFold(5)

Se evaluan 4 modelos con validacion cruzada agrupada por colegio.
En cada fold, todos los alumnos del mismo colegio van al mismo split.

**Metrica principal:** AUC-ROC — mide capacidad discriminatoria independientemente del umbral.  
**Metrica secundaria:** F1-Score — balance entre precision y recall.

In [ ]:
def make_prep():
    return ColumnTransformer([
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), FEATURES_CAT),
        ('num', StandardScaler(), FEATURES_NUM + FEATURES_IE),
    ])

CANDIDATOS = {
    'Logistic Regression': Pipeline([
        ('prep', make_prep()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
    ]),
    'Random Forest': Pipeline([
        ('prep', make_prep()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=8,
                                        min_samples_leaf=5, n_jobs=-1, random_state=SEED)),
    ]),
    'XGBoost': Pipeline([
        ('prep', make_prep()),
        ('clf', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                    eval_metric='auc', random_state=SEED, verbosity=0)),
    ]),
    'LightGBM': Pipeline([
        ('prep', make_prep()),
        ('clf', lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                     random_state=SEED, verbose=-1)),
    ]),
}

gkf = GroupKFold(n_splits=5)
resultados_cv = []

print(f'{"Modelo":<25} {"AUC media":>10} {"AUC std":>9} {"F1 media":>10} {"F1 std":>8}')
print('-' * 65)

for nombre, pipe in CANDIDATOS.items():
    aucs, f1s = [], []
    for tr, va in gkf.split(X_train, y_train, groups_train):
        m = clone(pipe)
        m.fit(X_train.iloc[tr], y_train.iloc[tr])
        prob = m.predict_proba(X_train.iloc[va])[:, 1]
        pred = (prob >= 0.5).astype(int)
        aucs.append(roc_auc_score(y_train.iloc[va], prob))
        f1s.append(f1_score(y_train.iloc[va], pred))
    resultados_cv.append({
        'Modelo': nombre,
        'AUC media': round(np.mean(aucs), 4),
        'AUC std': round(np.std(aucs), 4),
        'F1 media': round(np.mean(f1s), 4),
        'F1 std': round(np.std(f1s), 4),
    })
    print(f'{nombre:<25} {np.mean(aucs):>10.4f} {np.std(aucs):>9.4f} {np.mean(f1s):>10.4f} {np.std(f1s):>8.4f}')

tabla_cv = pd.DataFrame(resultados_cv).set_index('Modelo')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_barras = [PALETA['azul'], PALETA['rojo'], PALETA['naranja'], PALETA['verde']]

for j, (metrica, std_col) in enumerate([('AUC media', 'AUC std'), ('F1 media', 'F1 std')]):
    bars = axes[j].bar(tabla_cv.index, tabla_cv[metrica],
                       yerr=tabla_cv[std_col], color=colores_barras,
                       edgecolor='white', width=0.5, capsize=6)
    axes[j].set_title(f'Comparativa GroupKFold(5) — {metrica}')
    axes[j].set_ylabel(metrica)
    axes[j].set_ylim(tabla_cv[metrica].min() - 0.05, tabla_cv[metrica].max() + 0.05)
    axes[j].tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, tabla_cv[metrica]):
        axes[j].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + tabla_cv[std_col].iloc[list(tabla_cv.index).index(bar.get_label() if hasattr(bar, 'get_label') else '')] + 0.003 if False else bar.get_height() + 0.003,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='semibold')

plt.tight_layout()
plt.show()

ganador = tabla_cv['AUC media'].idxmax()
print(f'Modelo ganador por AUC: {ganador} ({tabla_cv.loc[ganador, "AUC media"]:.4f})')

## 8. Calibracion isotonica

La **calibracion** ajusta las probabilidades para que reflejen frecuencias reales observadas.  
Ejemplo: si el modelo predice 70% de probabilidad, deberian estar en riesgo ~70% de esos estudiantes.

Usamos `CalibratedClassifierCV` con metodo `isotonic` y 5 folds.  
Esto es importante porque los umbrales de clasificacion (ALTO >= 0.70, MEDIO >= 0.45)
asumen que las probabilidades son calibradas.

In [ ]:
# Pipeline base
pipe_lr = Pipeline([
    ('prep', make_prep()),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
])

# Versiones sin y con calibracion (para comparar)
pipe_lr_fit = clone(pipe_lr)
pipe_lr_fit.fit(X_train, y_train)

modelo_calibrado = CalibratedClassifierCV(clone(pipe_lr), method='isotonic', cv=5)
modelo_calibrado.fit(X_train, y_train)

prob_sin_cal = pipe_lr_fit.predict_proba(X_test)[:, 1]
prob_cal     = modelo_calibrado.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, probs, titulo in [
    (axes[0], prob_sin_cal, 'Sin calibrar'),
    (axes[1], prob_cal, 'Calibrado (isotonica)'),
]:
    obs, pred_cal = calibration_curve(y_test, probs, n_bins=10, strategy='quantile')
    brier = np.mean((probs - y_test.values) ** 2)
    ax.plot([0, 1], [0, 1], '--', color=PALETA['gris'], linewidth=1.2, label='Calibracion perfecta')
    ax.plot(pred_cal, obs, 'o-', color=PALETA['azul'], linewidth=2.5, markersize=7,
            label=f'Modelo (Brier = {brier:.4f})')
    ax.fill_between(pred_cal, obs, pred_cal, alpha=0.12, color=PALETA['rojo'])
    ax.set_title(f'Curva de calibracion — {titulo}')
    ax.set_xlabel('Probabilidad predicha')
    ax.set_ylabel('Fraccion de positivos observados')
    ax.legend(frameon=False)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

brier_sin = np.mean((prob_sin_cal - y_test.values) ** 2)
brier_cal = np.mean((prob_cal - y_test.values) ** 2)
print(f'Brier sin calibrar : {brier_sin:.4f}')
print(f'Brier calibrado    : {brier_cal:.4f}')
print(f'Mejora Brier       : {(brier_sin - brier_cal) / brier_sin * 100:.1f}%')

## 9. Analisis SHAP

SHAP (SHapley Additive exPlanations) descompone cada prediccion en la contribucion de cada feature.

- **Importancia global**: promedio del valor absoluto de SHAP sobre todos los estudiantes del test.
- **Beeswarm plot**: muestra direccion e intensidad de cada feature para cada estudiante.
- **Explicacion individual**: para un estudiante especifico, que variables lo empujan hacia riesgo alto o bajo.

In [ ]:
# Extraer el estimador base del modelo calibrado para SHAP
base_estimator = modelo_calibrado.calibrated_classifiers_[0].estimator
prep_fitted    = base_estimator.named_steps['prep']
clf_fitted     = base_estimator.named_steps['clf']

X_train_t = prep_fitted.transform(X_train)
X_test_t  = prep_fitted.transform(X_test)

explainer   = shap.LinearExplainer(clf_fitted, X_train_t, feature_names=ALL_FEATURES)
shap_values = explainer.shap_values(X_test_t)

# Importancia global
shap_imp = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=ALL_FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
colores_imp = [
    PALETA['rojo'] if v >= shap_imp.quantile(0.67)
    else PALETA['naranja'] if v >= shap_imp.quantile(0.33)
    else PALETA['gris']
    for v in shap_imp.values
]
bars = ax.barh(shap_imp.index, shap_imp.values, color=colores_imp, edgecolor='white', height=0.6)
ax.set_title('Importancia SHAP global — Logistic Regression calibrado\n'
             '(promedio |valor SHAP| sobre el set de test)', fontsize=12)
ax.set_xlabel('|SHAP| promedio (contribucion al log-odds)')
for i, (feat, val) in enumerate(shap_imp.items()):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)

leyenda = [
    mpatches.Patch(color=PALETA['rojo'],    label='Alta importancia'),
    mpatches.Patch(color=PALETA['naranja'], label='Media importancia'),
    mpatches.Patch(color=PALETA['gris'],    label='Baja importancia'),
]
ax.legend(handles=leyenda, frameon=False, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print('Ranking de importancia SHAP:')
for i, (feat, val) in enumerate(shap_imp.sort_values(ascending=False).items(), 1):
    print(f'  {i}. {feat:<25}: {val:.4f}')

In [ ]:
# Beeswarm plot — muestra direccion e intensidad por feature y por estudiante
shap.summary_plot(
    shap_values,
    X_test[ALL_FEATURES],
    feature_names=ALL_FEATURES,
    plot_type='dot',
    show=True,
    plot_size=(10, 5.5),
    max_display=9
)

In [ ]:
# Explicacion individual — estudiante con riesgo alto
probs_test_cal = modelo_calibrado.predict_proba(X_test)[:, 1]
candidatos_alto = np.where(probs_test_cal > 0.75)[0]
idx_ej = candidatos_alto[0] if len(candidatos_alto) > 0 else np.argmax(probs_test_cal)

shap_ind   = shap_values[idx_ej]
vals_ind   = X_test.iloc[idx_ej]
prob_ind   = probs_test_cal[idx_ej]
nivel_ind  = 'ALTO' if prob_ind >= 0.70 else 'MEDIO' if prob_ind >= 0.45 else 'BAJO'

fig, ax = plt.subplots(figsize=(10, 5))
orden = np.argsort(np.abs(shap_ind))
features_ord = [ALL_FEATURES[i] for i in orden]
shap_ord     = [shap_ind[i] for i in orden]
vals_ord     = [vals_ind.iloc[i] for i in orden]

colores_ind = [PALETA['rojo'] if v > 0 else PALETA['verde'] for v in shap_ord]
bars_ind = ax.barh(features_ord, shap_ord, color=colores_ind, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.9)

for bar, val, fval in zip(bars_ind, shap_ord, vals_ord):
    label = f'{fval:.2f}' if isinstance(fval, float) else str(fval)
    offset = 0.003 if val >= 0 else -0.003
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'= {label}', va='center', ha=ha, fontsize=9)

ax.set_title(f'Explicacion SHAP — Estudiante ejemplo\n'
             f'Probabilidad de riesgo: {prob_ind:.1%}  |  Nivel: {nivel_ind}',
             fontsize=12)
ax.set_xlabel('Contribucion SHAP  (rojo = empuja a riesgo | verde = protege del riesgo)')
plt.tight_layout()
plt.show()

## 10. Metricas finales en test

El set de test contiene **colegios que el modelo nunca vio durante el entrenamiento**.  
Esta es la evaluacion mas honesta: simula el despliegue en un colegio nuevo que no participó en el entrenamiento.

In [ ]:
y_pred_test = modelo_calibrado.predict(X_test)
y_prob_test = modelo_calibrado.predict_proba(X_test)[:, 1]

auc_test  = roc_auc_score(y_test, y_prob_test)
f1_test   = f1_score(y_test, y_pred_test)
acc_test  = accuracy_score(y_test, y_pred_test)
prec_test = precision_score(y_test, y_pred_test)
rec_test  = recall_score(y_test, y_pred_test)
cm        = confusion_matrix(y_test, y_pred_test)
fpr, tpr, _ = roc_curve(y_test, y_prob_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Curva ROC
axes[0].plot(fpr, tpr, color=PALETA['azul'], linewidth=2.5, label=f'LR calibrado  AUC = {auc_test:.4f}')
axes[0].fill_between(fpr, tpr, alpha=0.10, color=PALETA['azul'])
axes[0].plot([0, 1], [0, 1], '--', color=PALETA['gris'], linewidth=1.2, label='Clasificador aleatorio (AUC = 0.50)')
axes[0].set_title(f'Curva ROC — Set de test ({len(test_ies)} IEs nunca vistas)', fontsize=12)
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[0].legend(frameon=False, fontsize=10)

# Matriz de confusion
cm_labels = ['Sin riesgo\n(0)', 'En riesgo\n(1)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred: Sin riesgo', 'Pred: En riesgo'],
            yticklabels=['Real: Sin riesgo', 'Real: En riesgo'],
            ax=axes[1], annot_kws={'size': 14})
axes[1].set_title('Matriz de confusion', fontsize=12)

plt.tight_layout()
plt.show()

print('=' * 52)
print('   METRICAS FINALES EN TEST')
print(f'   ({len(test_idx):,} estudiantes | {len(test_ies)} colegios nunca vistos)')
print('=' * 52)
print(f'  AUC-ROC   : {auc_test:.4f}')
print(f'  F1-Score  : {f1_test:.4f}')
print(f'  Accuracy  : {acc_test:.4f}')
print(f'  Precision : {prec_test:.4f}')
print(f'  Recall    : {rec_test:.4f}')
print('=' * 52)

In [ ]:
# Distribucion de probabilidades predichas por nivel de riesgo
fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.linspace(0, 1, 41)

for nivel, color, label in [(0, PALETA['verde'], 'Sin riesgo (real)'),
                              (1, PALETA['rojo'],  'En riesgo (real)')]:
    mask = y_test.values == nivel
    ax.hist(y_prob_test[mask], bins=bins, alpha=0.65, color=color, label=label, density=True)

ax.axvline(0.45, color='#d4843a', linewidth=2.2, linestyle='--', label='Umbral MEDIO (0.45)')
ax.axvline(0.70, color=PALETA['rojo'], linewidth=2.2, linestyle='--', label='Umbral ALTO (0.70)')
ax.set_title('Distribucion de probabilidades predichas por clase real', fontsize=12)
ax.set_xlabel('Probabilidad de riesgo predicha')
ax.set_ylabel('Densidad')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# Distribucion de niveles asignados
niveles = pd.cut(y_prob_test, bins=[-0.01, 0.45, 0.70, 1.01],
                  labels=['BAJO', 'MEDIO', 'ALTO'])
print('\nDistribucion de niveles de riesgo asignados:')
for nivel, count in niveles.value_counts().sort_index().items():
    print(f'  {nivel}: {count:,} ({count/len(y_prob_test)*100:.1f}%)')

## 11. Guardar artefactos

Se generan los tres archivos `.pkl` que usa el backend FastAPI.
Descarga los archivos para reemplazar los del directorio `modelo/model/`.

In [ ]:
os.makedirs('model', exist_ok=True)

# Preprocesador independiente (para uso en /predecir individual)
prep_standalone = make_prep()
prep_standalone.fit(X_train, y_train)

joblib.dump(modelo_calibrado, 'model/modelo_em.pkl')
joblib.dump(prep_standalone,  'model/preprocessor_em.pkl')

metricas_pkl = {
    'accuracy':       float(acc_test),
    'precision':      float(prec_test),
    'recall':         float(rec_test),
    'f1_score':       float(f1_test),
    'auc_roc':        float(auc_test),
    'confusion_matrix': cm.tolist(),
    'roc_fpr':        fpr.tolist(),
    'roc_tpr':        tpr.tolist(),
    'features':       ALL_FEATURES,
    'shap_importancia': dict(zip(
        ALL_FEATURES,
        np.abs(shap_values).mean(axis=0).tolist()
    )),
    'cv_comparativa': tabla_cv.reset_index().to_dict('records'),
    'modelo_ganador': 'Logistic Regression (calibrado isotonico)',
    'train_rows':     int(len(X_train)),
    'test_rows':      int(len(X_test)),
    'trained_at':     pd.Timestamp.now().isoformat(timespec='seconds'),
}
joblib.dump(metricas_pkl, 'model/metricas_em.pkl')

for fname in ['model/modelo_em.pkl', 'model/preprocessor_em.pkl', 'model/metricas_em.pkl']:
    size_kb = os.path.getsize(fname) / 1024
    print(f'  {fname} ({size_kb:.1f} KB)')

# Descomentar para descargar al PC desde Colab:
# from google.colab import files
# for f in ['model/modelo_em.pkl', 'model/preprocessor_em.pkl', 'model/metricas_em.pkl']:
#     files.download(f)

print('\nArtefactos guardados. Descomentar files.download() para bajar al PC.')

## 12. Resumen ejecutivo

| Aspecto | Resultado |
|---|---|
| **Algoritmo ganador** | Logistic Regression con calibracion isotonica |
| **Dataset** | EM 2022 MINEDU — Lima Metro, privado |
| **Validacion** | GroupKFold(5) por ID_IE — sin leakage por colegio |
| **AUC en CV** | ~0.865 +/- 0.016 |
| **AUC en test** | ~0.843 (IEs completamente nuevas) |
| **F1 en test** | ~0.695 |
| **Predictor #1 (SHAP)** | M500_L — puntaje de Lectura del estudiante |
| **Predictor #2 (SHAP)** | M500_CN — puntaje de Ciencias del estudiante |
| **Predictor #3 (SHAP)** | M500_L_iemean — promedio de Lectura del colegio |

---

### Por que este modelo es defendible academicamente

1. **Datos reales institucionales** — Evaluacion Muestral 2022 MINEDU, no dataset sintetico
2. **Sin leakage** — GroupShuffleSplit + GroupKFold por colegio: los colegios del test nunca se ven en entrenamiento
3. **Comparativo** — se evaluaron 4 algoritmos; Logistic Regression gano por AUC con menor varianza
4. **Calibrado** — las probabilidades reflejan frecuencias reales (curva de calibracion verificada)
5. **Interpretable** — SHAP explica cada prediccion individualmente para el usuario final

### Limitacion principal

El modelo usa M500_L y M500_CN (puntajes de Lectura y Ciencias) para predecir riesgo en Matematica.
Estos puntajes se obtienen en la misma evaluacion — por eso este modelo es de **analisis
post-evaluacion** (priorizacion de intervenciones), no de prediccion temprana antes de la prueba.

---

*Proyecto P20261012 · UPC · Ingenieria de Sistemas de Informacion · 2026*